# ScholarPulse — 8-Pass Ingest Pipeline (Colab Production)

Production notebook for **pass0 → pass7** E2E ingest with:

- Google Drive checkpoint resume (`stage.sqlite` + `cursor.json`)
- GROBID hybrid routing (`SP_GROBID_MODE=auto`: clean PDF → GROBID, dirty → regex)
- Kaggle failover notes (see last cell)

**Manual gate remaining:** run once on Colab T4 GPU with real weights → confirm `algorithm_version=v1-colab-ml` and bulk POST.

In [ ]:
# 1) Clone repo + deps (adjust branch/path as needed)
!git clone -q https://github.com/your-org/scholarpulse-final.git /content/scholarpulse-final 2>/dev/null || true
%cd /content/scholarpulse-final

!pip -q install requests pymupdf faiss-cpu sentence-transformers transformers
!pip -q install torch --index-url https://download.pytorch.org/whl/cu118

import sys
from pathlib import Path
ROOT = Path('/content/scholarpulse-final')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# 2) Mount Drive + checkpoint env
from google.colab import drive
drive.mount('/content/drive')

import os
RUN_ID = 'run-001'
os.environ.update({
    'SP_USE_REAL_MODELS': '1',
    'SP_GROBID_MODE': 'auto',
    'SP_CHECKPOINT_DIR': f'/content/drive/MyDrive/scholarpulse/checkpoints/{RUN_ID}',
    'SP_CHECKPOINT_EVERY': '25',
    'SP_COMPUTE_PLATFORM': 'colab',
    'SP_INGEST_API_URL': 'https://YOUR-WORKER/api/cite/bulk-ingest',
})
print('checkpoint:', os.environ['SP_CHECKPOINT_DIR'])

In [ ]:
# 3) GPU check + optional GROBID Docker
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

# !docker pull -q lfoppiano/grobid:0.8.0
# !docker run -d --rm -p 8070:8070 --name grobid lfoppiano/grobid:0.8.0

In [ ]:
!python -m colab.pipeline.smoke_test

In [ ]:
MANIFEST = '/content/drive/MyDrive/scholarpulse/manifest.json'

!python colab/notebooks/ingest_pipeline_runner.py \
  --manifest {MANIFEST} \
  --checkpoint-dir /content/drive/MyDrive/scholarpulse/checkpoints \
  --run-id {RUN_ID} \
  --out-dir /content/payloads \
  --ingest

## Kaggle failover ($0 backup)

1. Upload notebook → Kaggle **Settings → Accelerator → GPU T4 x2**
2. Mount/copy Drive checkpoint `checkpoints/{run_id}/`
3. Re-run `ingest_pipeline_runner.py` (skips DOIs with status `ok` in `stage.sqlite`)
4. Respect **30 GPU h/week**; Colab Free fills remainder

```bash
export SP_COMPUTE_PLATFORM=kaggle
export SP_GROBID_MODE=auto
export SP_RESUME_CHECKPOINT=/kaggle/working/checkpoints/run-001/stage.sqlite
```